<a href="https://colab.research.google.com/github/aliffiaaps-coder/Unsupervised-Learning-Clustering/blob/main/Aliffia_Azzahra_Putri_S_Clustering_WEEK_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STUDY CASE UNSUPERVISED LEARNING CLUSTERING**

Nama  : Aliffia Azzahra Putri S.<br>
Batch : 65


In [ ]:
# Import library yang digunakan
import os
import warnings
import pandas as pd

# Ignore warning
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive',force_remount=True)

# 1. Proses ekstrak data
data_flight = pd.read_csv("/content/gdrive/MyDrive/Colab Notebooks/WEEK 14/flight_train.csv")

# 2. Ubah nama kolom menjadi lowercase semua
data_flight.columns = data_flight.columns.str.lower()

# 3. Tampilkan sampel data
display(data_flight.head())

In [ ]:
data_flight = data_flight.sample(5000)

In [ ]:
data_flight.info()

In [ ]:
# Mengubah type member_no menjadi object
data_flight['member_no'] = data_flight['member_no'].astype(str)

data_flight.info()

#### **🔍 Insight**
Kolom `member_no` diubah ke tipe 'object' karena merupakan nilai unik yang mendefinisikan suatu identitas seseorang.

## **1. Proses Exploratory Data Analysis (EDA) & Preprocessing**

#### **1A. Statistika Deskriptif**

In [ ]:
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# Memahami data dengan Statistika Deskriptif pada kolom numerik
data_flight.describe()

In [ ]:
# Memahami data dengan pada kolom kategori
data_flight.describe(include='object')

In [ ]:
# Memahami sebaran data pada kolom kategorik
# Definisikan kolom non numerik
kolom_kategorik = data_flight.select_dtypes(include=['object', 'string']).columns

# Tampilkan hasilnya
print(kolom_kategorik)

In [ ]:
# Statistik Deskriptif kolom Kategorik (non-numerik)
for col in kolom_kategorik:
    print(f'> Frekuensi \033[93m{col}\033[0m')
    print(f'  Terdapat {data_flight[col].nunique()} data unik\n')
    df_frekuensi = data_flight[col].value_counts(dropna=False).reset_index(name='count')
    display(df_frekuensi)
    print('\n')

#### **1B. Cek Missing Value**

In [ ]:
# Tampilkan missing value pada data_flight
display(data_flight.isna().sum())

In [ ]:
# Mendefinisikan total missing value pada data

data_null = data_flight.isnull().sum().reset_index()
data_null.columns = ['feature','missing_value']
data_null['percentage'] = round((data_null['missing_value']/len(data_flight))*100,2)
data_null = data_null.sort_values('percentage', ascending=False).reset_index(drop=True)
data_null = data_null[data_null['percentage']>0]
data_null

#### **🔍 Insight: Missing Value**
1. Missing value pada kolom numerik : `age`, `sum_yr_1`, dan `sum_yr_2`
2. Missing value pada kolom kategorik : `work_province`, `work_city`, dan `work_province`<br>

Perlu dilakukan tindakan lebih lanjut untuk mengatasi missing value di atas.

#### **1C. Handle Missing Value**

In [ ]:
# Handle Missing Value Kolom Numerik
# Mengisi missing value pada kolom 'age' dengan nilai Median
median_age = data_flight['age'].median()
data_flight['age'] = data_flight['age'].fillna(median_age)

# Mengisi missing value pada kolom 'sum_yr_1' dengan nilai Median
median_yr_1 = data_flight['sum_yr_1'].median()
data_flight['sum_yr_1'] = data_flight['sum_yr_1'].fillna(median_yr_1)

# Mengisi missing value pada kolom 'sum_yr_2' dengan nilai Median
median_yr_2 = data_flight['sum_yr_2'].median()
data_flight['sum_yr_2'] = data_flight['sum_yr_2'].fillna(median_yr_2)

# Tampilkan kembali missing value pada data numerik
display(data_flight.select_dtypes(include=['int64', 'float64']).isna().sum())

In [ ]:
# Melihat nilai Modus pada data kategorik
print(data_flight['work_country'].value_counts(normalize=True).head())
print(data_flight['work_province'].value_counts(normalize=True).head())
print(data_flight['work_city'].value_counts(normalize=True).head())

In [ ]:
# Handle Missing Value Kolom Kategorik
# Mengisi kolom teks yang kosong dengan kategori 'Unknown'
data_flight['work_city'] = data_flight['work_city'].fillna('Unknown')

data_flight['work_province'] = data_flight['work_province'].fillna('Unknown')

modus_country = data_flight['work_country'].mode()[0]
data_flight['work_country'] = data_flight['work_country'].fillna(modus_country)

In [ ]:
# Tampilkan kembali missing value pada data kategorik
display(data_flight.select_dtypes(include=object).isna().sum())


In [ ]:
data_flight.info()


#### **🔍 Insight: Handle Missing Value**

1. Mengatasi missing value pada kolom `age`, `sum_yr_1`, dan `sum_yr_2` dengan **Median**. Nilai median digunakan karena lebih aman dan tahan terhadap banyak pencilan/outlier.
2. Missing value pada kolom kategorik `work_country` diisi dengan Modus. Imputasi menggunakan Modus pada kolom ini dilakukan dengan asumsi bahwa data yang hilang berdistribusi mengikuti pola populasi terbesar (negara mayoritas). Langkah ini diambil untuk menghindari bias preferensi kelompok minoritas, mencegah kehilangan informasi penting (data loss) akibat penghapusan baris, serta menjaga konsistensi karakteristik sebaran pelanggan sebelum memasuki tahap pembentukan fitur seleksi.
3. Missing value pada kolom kategorik `work_city`, `work_province` diisi dengan 'Unknown'. Kolom diisi dengan 'unknown'. Karena kolom tersebut mendeskripsikan profil masing-masing penumpang. Jika menggunakan modus, kolom akan memaksa bahwa setiap individu berasal dari tempat tersebut, padahal nyatanya tidak demikian. Hal ini bisa mengganggu analisis bisnis yang kurang tepat karena data yang kosong tidak memberikan informasi yang sebenarnya terhadap penumpang. Mengisi dengan 'Unknown' menjaga data tetap objektif dan memberikan karakteristik yang jelas dalam melakukan model clustering.

#### **1D. Cek Duplikat Data**

In [ ]:
data_flight.duplicated().sum()

In [ ]:
data_flight['member_no'].duplicated().sum()

#### **🔍 Insight: Duplikat Data**
Tidak ada data yang dupilkat sehingga tidak perlu handle duplicate.

#### **1E. Distribusi Data & Cek Outlier**

In [ ]:
# Histogram distribusi data Kolom Numerik
import seaborn as sns

h = data_flight.hist(bins=25,figsize=(16,16),xlabelsize='10',ylabelsize='10',xrot=-15)
sns.despine(left=True, bottom=True)
[x.title.set_size(12) for x in h.ravel()];
[x.yaxis.tick_left() for x in h.ravel()];

In [ ]:
feat_num = list(data_flight)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Box plot untuk mengetehaui nilai outliers
# Mendefinisikan kolom numerikal dari data_flight DataFrame
numerical_cols = data_flight.select_dtypes(include=['int64', 'float64']).columns.tolist()

num_features_to_plot = len(numerical_cols)

# Menentukan ukuran grid
ncols = 4  # Jumlah kolom pada grid subplot
nrows = math.ceil(num_features_to_plot / ncols)

plt.figure(figsize=(20, 5 * nrows))

for i, feature in enumerate(numerical_cols):
    # Membuat subplot: (nrows, ncols, current_plot_index)
    plt.subplot(nrows, ncols, i + 1)
    sns.boxplot(y=data_flight[feature], color='green', orient='v')
    plt.title(feature)
    plt.ylabel('')

plt.tight_layout()
plt.show()

#### **🔍 Insight: Outlier**
Berdasarkan grafik histogram sebaran data, hampir seluruh kolom aktivitas penerbangan didominasi oleh pola outlier. Kolom-kolom tersebut meliputi:
1. **sum_yr_1 & sum_yr_2** (Pendapatan tahunan dari pelanggan)
2. **seg_km_sum** (Total jarak tempuh dalam KM)
3. **flight_count** (Frekuensi terbang)
4. **exchange_count** (Penukaran poin)
5. **points_sum** **& point_notflight** (Akumulasi poin loyalitas)

Mayoritas penumpang (90%+) adalah penumpang biasa yang jarang terbang dengan pengeluaran kecil. Namun, titik-titik yang memanjang jauh ke kanan tersebut adalah outlier yang berisi para penumpang VIP/Sultan yang mana jumlahnya sedikit, tetapi memberikan kontribusi finansial yang besar bagi maskapai.

Dalam analisis segmentasi maskapai ini, **outlier tetap dipertahankan**(tidak dihapus). Hal ini dikarenakan pencilan tersebut dpaat menjadi representasi dari kelompok pelanggan bernilai tinggi (high-value customers/frequent flyers) yang menjadi target utama dalam pemodelan klasterisasi LRFMC."

#### **1F. Analisis Korelasi**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Mengambil hanya kolom numerik
data_numerik = data_flight.select_dtypes(include=['int64', 'float64'])

# 2. Menghitung matriks korelasi
corr_ = data_numerik.corr()

# 3. Membuat Heatmap menggunakan Seaborn
plt.figure(figsize=(16, 10))
sns.heatmap(
    corr_,
    annot=True,
    fmt=".2f",
    cmap="BuPu",
    linewidths=0.5,
    cbar=True
)

plt.title("Matriks Korelasi Kolom Numerik - Data Flight", fontsize=16, fontweight='bold', pad=20)

# 4. Menampilkan grafik
plt.tight_layout() # Agar teks nama kolom di bawah/samping tidak terpotong
plt.show()

#### **🔍Insight : Analisis Korelasi**

**1. Korelasi Sangat Kuat**
- Fitur-fitur seperti ***flight_count*** (jumlah terbang), **bp_sum** (poin rencana perjalanan), **sum_yr_1/sum_yr_2** (akumulasi pendapatan), dan **seg_km_sum** (total jarak tempuh) memiliki nilai korelasi positif antara 0.75 hingga 0.92.
- Ini membuktikan bahwa pelanggan yang paling sering terbang (flight_count) adalah mereka yang memberikan kontribusi jarak terbang terjauh sekaligus menyumbang pendapatan terbesar bagi maskapai. Perilaku ini sangat linear.

**2. Korelasi Negatif**
- Kolom **last_to_end** (jarak hari sejak penerbangan terakhir) memiliki korelasi negatif (sekitar -0.32 hingga -0.43) dengan kolom jumlah terbang dan total jarak tempuh.
- Semakin kecil angka last_to_end, artinya seseorang tersebut baru saja terbang belum lama ini, maka total frekuensi terbang dan akumulasi jaraknya sepanjang tahun semakin tinggi. Ini adalah indikator penting untuk mengenali pelanggan aktif dan pelanggan pasif.

**3. Korelasi Lemah**
- Kolom **age** (umur) memiliki korelasi yang mendekati nol (-0.07 hingga 0.10) dengan hampir semua fitur lainnya. Artinya fitur yang tersebut Tidak Berpengaruh.
- Umur seseorang tidak menentukan seberapa sering atau seberapa jauh mereka akan terbang menggunakan maskapai ini. Penumpang muda maupun tua memiliki peluang perilaku terbang yang sama rata.

# **2. Feature Engineering**

Untuk industri penerbangan, model segmentasi pelanggan yang umum digunakan secara bisnis adalah model LRFMC (*Length, Recency, Frequency, Monetary, Discount*). Oleh karena itu, model tersebut digunakan dalam penerapan Model Clustering dengan beberapa fitur yang dipilih. Penjelasannya adalah sebagai berikut:

- **L (Length / Hubungan):** Dihitung dari selisih load_time (waktu observasi data) dengan ffp_date (tanggal bergabung keanggotaan).<br>
-> Mengetahui seberapa setia (loyal) pelanggan telah menjadi member maskapai.

- **R (Recency / Kebaruan)**: Diambil dari kolom last_to_end (jarak hari dari penerbangan terakhir ke waktu observasi atau ke pesanan penerbangan paling akhir).<br>
-> Mengetahui kapan terakhir kali pelanggan menggunakan jasa penerbangan perusahaan. Jika nilainya sangat besar, pelanggan berpotensi telah pindah ke maskapai kompetitor.

- **F (Frequency / Frekuensi)**: Diambil dari kolom flight_count (total berapa kali terbang).<br>
-> Menunjukkan tingkat keaktifan pelanggan. Semakin tinggi frekuensinya, semakin sering pelanggan menggunakan maskapai perusahaan.

- **M (Monetary / Nilai Finansial)**: Diambil dari kolom seg_km_sum (total jarak penerbangan dalam kilometer yang sudah ditempuh).<br>
-> Dalam bisnis maskapai, total jarak tempuh berbanding lurus dengan jumlah uang yang pelanggan belanjakan. Ini mengidentifikasi pelanggan bernilai tinggi (VIP).<br>

- **C (Discount / Rasio Diskon)**: Diambil dari kolom avg_discount (rata-rata rasio diskon yang digunakan pelanggan).<br>
-> Membantu perusahaan mengetahui sensitivitas harga pelanggan. Apakah mereka tipe penyuka diskon atau pelanggan yang rela membayar harga penuh.

#### **2A. Feature Engineering & Feature Selection**

In [ ]:
# Memulai Proses Feature Engineering (Model LRFMC)

# 1. Memastikan kolom tanggal bertipe Datetime
data_flight['load_time'] = pd.to_datetime(data_flight['load_time'])
data_flight['ffp_date'] = pd.to_datetime(data_flight['ffp_date'])

# 2. Membuat DataFrame baru khusus untuk fitur clustering
df_lrfmc = pd.DataFrame()

# Feature 1: L (Length) -> Selisih tanggal dalam satuan BULAN
df_lrfmc['L'] = (data_flight['load_time'] - data_flight['ffp_date']).dt.days / 30.0

# Feature 2: R (Recency) -> Diambil dari 'last_to_end'
df_lrfmc['R'] = data_flight['last_to_end']

# Feature 3: F (Frequency) -> Diambil dari 'flight_count'
df_lrfmc['F'] = data_flight['flight_count']

# Feature 4: M (Monetary) -> Diambil dari 'seg_km_sum' (Jarak Tempuh)
df_lrfmc['M'] = data_flight['seg_km_sum']

# Feature 5: C (Discount) -> Diambil dari 'avg_discount'
df_lrfmc['C'] = data_flight['avg_discount']

# 4. Menampilkan Hasil Akhir Feature Engineering
print("Hasil Feature Engineering")
display(df_lrfmc.head())

In [ ]:
# 5. Menampilkan Ringkasan Statistik Fitur Baru
print("Ringkasan Statistik Fitur LRFMC")
display(df_lrfmc.describe())

#### **🔍 Insight: Feature Engineering & Selection**
Dalam menentukan karakteristik segmentasi pelanggan maskapai penerbangan, fitur yang digunakan tidak seluruhnya guna menghindari efek multikolinieritas. Fitur ini diekstraksi dan dirangkum menjadi 5 Fitur **(Length, Recency, Frequency, Monetary, dan Discount).** Fitur ini dikenal sebagai LRFMC, digunakan untuk menganalisis dan melakukan segmentasi (pengelompokan) pelanggan berdasarkan nilai serta riwayat transaksi.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Menghitung korelasi antar fitur LRFMC
corr_lrfmc = df_lrfmc.corr()

# 2. Mengatur ukuran kanvas grafik
plt.figure(figsize=(10, 8))

# 3. Membuat Heatmap Korelasi
sns.heatmap(
    corr_lrfmc,
    annot=True,
    fmt=".2f",
    cmap="BuPu",
    linewidths=1,
    square=True
)

# 4. Memberikan Judul
plt.title("Matriks Korelasi Fitur LRFMC", fontsize=14, fontweight='bold', pad=15)

# 5. Menampilkan Grafik
plt.tight_layout()
plt.show()

#### **2B. Scaling**

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Inisialisasi objek StandardScaler
scaler = StandardScaler()

# 2. Melakukan fit dan transform pada data LRFMC
# fit_transform akan menghitung rata-rata & standar deviasi sekaligus mengubah nilainya
data_scaled_array = scaler.fit_transform(df_lrfmc)

# 3. Mengembalikan hasil scaling (yang berbentuk array) menjadi DataFrame Pandas agar rapi
df_lrfmc_scaled = pd.DataFrame(data_scaled_array, columns=df_lrfmc.columns)

print("Menampilkan data setelah scaling")
display(df_lrfmc_scaled.head())

In [ ]:
print("Statistik Deskriptif Setelah Scaling ---")
# Menampilkan deskripsi untuk memastikan mean 0 dan std = 1
display(df_lrfmc_scaled.describe())

# **3. Clustering**

#### **3A.Elbow Method**

Elbow method : untuk menentukan jumlah cluster yang optimal

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# 1. Menghitung inertia untuk jumlah cluster 1 sampai 10
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(df_lrfmc_scaled)
    wcss.append(kmeans.inertia_)

# 2. Membuat grafik Elbow Method
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--', color='b')
plt.title('What is the Best Number for KMeans ?', fontsize=14, fontweight='bold')
plt.xlabel('Jumlah Cluster (k)')
plt.ylabel('WCSS / Inertia')
plt.xticks(range(1, 11))
plt.grid(True)
plt.show()

#### **🔍 Insight**

Grafik mulai melandai tajam terlihat pada angka 3 sehingga didapatkan jumlah cluster yang optimal -> **k=3**

#### **3B. K-Means**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# 1. K-MEANS VALID
k_optimal = 3
kmeans = KMeans(n_clusters=k_optimal, init='k-means++', random_state=42)

# Ambil data murni (buang kolom cluster jika tidak sengaja sudah ada di dalam data)
X_scaled = df_lrfmc_scaled.drop(columns=['cluster_kmeans', 'cluster_agg'], errors='ignore')

# Fitting menggunakan data X_scaled yang sudah dijamin bersih
df_lrfmc_scaled['cluster_kmeans'] = kmeans.fit_predict(X_scaled)

# Menghitung silhouette score secara objektif
score_kmeans = silhouette_score(X_scaled, df_lrfmc_scaled['cluster_kmeans'])

print("HASIL K-MEANS")
print(f"Silhouette Score K-Means : {score_kmeans:.4f}\n")

#### **3C. Agglomerative Clustering**

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# 2. AGGLOMERATIVE VALID
k_agglom = 3
agg_model = AgglomerativeClustering(n_clusters=k_agglom, linkage='ward')

# Ambil data murni (buang kolom cluster agar tidak mengacaukan perhitungan jarak)
X_scaled = df_lrfmc_scaled.drop(columns=['cluster_kmeans', 'cluster_agg'], errors='ignore')

# Fitting dan simpan hasilnya ke kolom 'cluster_agg'
df_lrfmc_scaled['cluster_agg'] = agg_model.fit_predict(X_scaled)

# Menghitung silhouette score secara objektif
score_agg = silhouette_score(X_scaled, df_lrfmc_scaled['cluster_agg'])

print("HASIL AGGLOMERATIVE")
print(f"Silhouette Score Agglomerative : {score_agg:.4f}")

#### **🔍Insight: K-Means & Agglomerative**
**Model K-Means Clustering** dipilih sebagai **model terbaik** karena memiliki nilai **Silhouette Score yang lebih tinggi (0.27)**. Hal ini membuktikan bahwa K-Means mampu mengelompokkan data pelanggan LRFMC maskapai secara lebih padat di dalam klasternya sendiri, dan memberikan batas pemisahan yang lebih jelas antar-kelompok dibandingkan metode hierarki Agglomerative.

#### **3D. Visualisasi PCA**

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA

# Memproses Reduksi Dimensi dengan PCA

# 1. Mengisolasi fitur murni (menghapus label cluster agar tidak ikut dihitung PCA)
fitur_scaled = df_lrfmc_scaled.drop(columns=['cluster_kmeans', 'cluster_agg'], errors='ignore')

# 2. Menjalankan PCA untuk mereduksi menjadi 2 Komponen Utama
pca = PCA(n_components=2, random_state=42)
pca_data = pca.fit_transform(fitur_scaled)

# 3. Memasukkan hasil koordinat PCA ke dalam DataFrame baru bersama label klaster masing-masing
df_pca = pd.DataFrame(data=pca_data, columns=['PCA 1', 'PCA 2'])
df_pca['cluster_kmeans'] = df_lrfmc_scaled['cluster_kmeans'].values
df_pca['cluster_agg'] = df_lrfmc_scaled['cluster_agg'].values

print(f"PCA Selesai! Data siap divisualisasikan (Ukuran: {df_pca.shape})")



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Membuat Plot Visualisasi Berdampingan ---")
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Grafik Kiri: K-Means
sns.scatterplot(x='PCA 1', y='PCA 2', hue='cluster_kmeans', data=df_pca, palette='Set1', alpha=0.7, edgecolor='k', ax=axes[0])
axes[0].set_title(f'Visualisasi Cluster K-Means\n(Silhouette Score: {score_kmeans:.4f})', fontsize=14, fontweight='bold', pad=10)
axes[0].set_xlabel('Principal Component 1')
axes[0].set_ylabel('Principal Component 2')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Grafik Kanan: Agglomerative
sns.scatterplot(x='PCA 1', y='PCA 2', hue='cluster_agg', data=df_pca, palette='Set2', alpha=0.7, edgecolor='k', ax=axes[1])
axes[1].set_title(f'Visualisasi Cluster Agglomerative\n(Silhouette Score: {score_agg:.4f})', fontsize=14, fontweight='bold', pad=10)
axes[1].set_xlabel('Principal Component 1')
axes[1].set_ylabel('Principal Component 2')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle(f'Perbandingan Hasil Clustering Model LRFMC (Data Sample: {len(df_pca)} Baris)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# **4. Interpretasi Cluster**

#### **4A. Profiling Cluster**

In [ ]:
# Memasukkan label cluster K-Means ke DataFrame nilai asli
df_lrfmc['cluster_kmeans'] = df_pca['cluster_kmeans'].values

# Menghitung rata-rata nilai asli LRFMC untuk setiap cluster
df_profile = df_lrfmc.groupby('cluster_kmeans').mean()
df_profile['Jumlah_Pelanggan'] = df_lrfmc.groupby('cluster_kmeans').size()
df_profile['Persentase_%'] = (df_profile['Jumlah_Pelanggan'] / len(df_lrfmc)) * 100

print("TABEL INTERPRETASI & REKOMENDASI BISNIS")
display(df_profile.round(2))

#### **4B. Insight: Profiling Cluster**

Berdasarkan hasil analisis statistik fitur LRFMC, pelanggan maskapai berhasil dikelompokkan secara spesifik menjadi **3 karakteristik perilaku** (*customer behavior*) berikut:


- **Cluster 0 (Churn Risk)**: Klaster ini diisi oleh pelanggan pasif yang berisiko tinggi telah berpindah ke maskapai kompetitor. Indikator utamanya terlihat dari nilai *Recency* yang sangat ekstrem (R = 432.52 hari), yang berarti rata-rata dari mereka sudah **lebih dari satu tahun tidak pernah terbang lagi** menggunakan maskapai ini. Frekuensi terbang (F = 4.81 kali) dan jarak tempuh mereka (M = 7.741,85 KM) juga merupakan yang paling rendah.<br>
--> **Porsi Pasar:** Meliputi **1.117 pelanggan (22.34%)** dari total data sampel.

- **Cluster 1 (Potential Loyalists)**: Klaster ini merupakan kelompok pelanggan umum atau kelas menengah yang cukup aktif melakukan penerbangan. Mereka memiliki masa keanggotaan moderat (L = 50.11 bulan) dan masih rutin terbang dalam beberapa bulan terakhir (R = 85.85 hari). Frekuensi terbang (F = 10.87 kali) dan total jarak tempuh mereka (M = 15.579 KM) berada di tingkat menengah.<br>
--> **Porsi Pasar:** Merupakan kelompok terbesar dengan total **3.342 pelanggan (66.84%)**. Pelanggan ini sangat potensial untuk diberikan promosi agar meningkatkan intensitas penerbangannya di masa depan.

- **Cluster 2 (VIP)** : Klaster ini diisi oleh kelompok pelanggan premium atau *frequent flyers* kelas atas. Mereka memiliki masa keanggotaan paling lama (L = 66.10 bulan) dan sangat aktif melakukan perjalanan akhir-akhir ini (R = 27.09 hari). Karakteristik utama mereka adalah frekuensi terbang yang sangat tinggi (F = 45.14 kali) dengan total jarak tempuh terjauh (M = 66.423,25 KM). <br>
--> **Porsi Pasar:** Kelompok ini berjumlah **541 pelanggan (10.82%)**. Meskipun minoritas, mereka adalah kelompok yang paling banyak menyumbang keuntungan finansial terbesar bagi perusahaan.


#### **4C. Rekomendasi Bisnis**

Berikut marketing berbasis personalisasi pelanggan sebagai **rekomendasi bisnis kepada manajemen maskapai**:

1. **Strategi untuk Cluster Churn Risk**<br>
Luncurkan kampanye aktivasi kembali (Win-Back Strategy). Gunakan automated email marketing dengan subjek yang menarik empati (contoh: "Kami merindukan Anda di udara, berikut diskon khusus 25% untuk penerbangan Anda berikutnya"). Tambahkan survei singkat satu pertanyaan untuk mengetahui alasan mereka berhenti menggunakan maskapai (apakah karena harga, ketepatan waktu, atau rute).
- Tujuan: Memancing kembali transaksi pertama setelah sekian lama tidak aktif agar siklus keanggotaannya kembali bernilai.


2. **Strategi untuk Cluster Potential Loyalists**<br>
Tingkatkan loyalitas menggunakan metode gamification (Upselling Strategy). Kirimkan promo terpersonalisasi melalui aplikasi dengan contoh: "Terbang 3 kali lagi dalam kurun waktu 3 bulan ke depan untuk mendapatkan status keanggotaan Gold dan bonus 10.000 poin". Kemudian lakukan edukasi untuk kemudahan penukaran poin penerbangan dengan voucer hotel atau restoran mitra.
- Tujuan: Mendorong kelompok mayoritas ini agar meningkatkan frekuensi terbangnya dan perlahan naik kelas menjadi Cluster VIP.

3. **Strategi untuk Cluster VIP**<br>
Untuk Cluster ini, implementasikan program apresiasi eksklusif (Retain Strategy). Berikan fasilitas jalur bagasi cepat (Priority Baggage), akses gratis ke airport lounge, kuota bagasi ekstra, serta opsi upgrade kelas penerbangan gratis jika kursi kelas bisnis masih tersedia menjelang boarding.
- Tujuan: Menjaga kepuasan tertinggi agar mereka tidak berpindah ke program loyalitas maskapai pesaing.